Carleton School of Information Technology

Final project: A simple setup to train DistilBERT with optimized hyperparameters

ITEC 5920 – Applied Deep Learning Winter 2026

Student: Michael Rolbin Student #100865849

## 1. Imports

In [ ]:
import re
import json
import warnings
import logging
import itertools
from pathlib import Path
from typing import Dict, List, Tuple

import glob
from huggingface_hub import snapshot_download
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
import logging as _logging
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW

from datasets import load_dataset, DatasetDict
from huggingface_hub import snapshot_download
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    set_seed,
    DataCollatorWithPadding,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, roc_curve,
)

warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)

## 2. Global Parameters

In [ ]:
SEED = 42
set_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

plt.style.use('seaborn-v0_8-whitegrid')
MODEL_COLORS = {
    'DistilBERT': '#C44E52',
}

## 3. DistilBERT Hyperparameter Configuration

A single fixed hyperparameter configuration is used for DistilBERT training.

| HP | DistilBERT | Rationale |
|---|---|---|
| `max_length` | 128 | Context vs. speed tradeoff |
| `batch_size` | 32 | Memory vs. throughput |
| `num_epochs` | 4 | Convergence budget |
| `learning_rate` | 3e-5 | Architecture sensitivity |
| `weight_decay` | 0.00 | Regularisation |
| `warmup_ratio` | 0.06 | LR warmup fraction |
| `gradient_clip` | 1.0 | Gradient stability |


In [ ]:
MODEL_HF_IDS = {
    'DistilBERT': 'distilbert-base-uncased',
}

# Single fixed hyperparameter configuration for DistilBERT
MODEL_HP_GRIDS: Dict[str, Dict] = {
    'DistilBERT': {
        'max_length':    [128],
        'batch_size':    [32],
        'num_epochs':    [4],
        'learning_rate': [3e-5],
        'weight_decay':  [0.00],
        'warmup_ratio':  [0.06],
        'gradient_clip': [1.0],
    },
}


## 4. Data Loading & Preprocessing

In [ ]:
# Download parquet files for the dataset
snapshot_dir = snapshot_download(
    repo_id='neuralchemy/Prompt-injection-dataset',
    repo_type='dataset',
    allow_patterns='*.parquet',
)
parquet_files = sorted(glob.glob(f'{snapshot_dir}/**/*.parquet', recursive=True))
print(f'Found {len(parquet_files)} parquet file(s):')
for p in parquet_files:
    print(f'  {p}')

In [ ]:
# Read each parquet file individually, normalise column names, and concatenate
KEEP_COLS = ['id', 'text', 'label', 'domain', 'subdomain']

frames = []
for path in parquet_files:
    df_part = pd.read_parquet(path)

    # Rename 'combined_text' to 'text'
    if 'combined_text' in df_part.columns and 'text' not in df_part.columns:
        df_part = df_part.rename(columns={'combined_text': 'text'})

    cols = [c for c in KEEP_COLS if c in df_part.columns]
    frames.append(df_part[cols])

df_raw = pd.concat(frames, ignore_index=True)
print(f'\nTotal rows        : {len(df_raw)}')
print(f'Columns           : {list(df_raw.columns)}')
print(f'{df_raw["label"].value_counts().rename({0: "Benign", 1: "Harmful"}).to_string()}')

In [ ]:
# NaN cleaning
df_no_nan = df_raw.dropna(subset=['text', 'label']).reset_index(drop=True)

def clean_text(text) -> str:
    if text is None or (isinstance(text, float) and (text != text)):
        return ''
    if not isinstance(text, str):
        text = str(text)
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    return text

def preprocess_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['text']  = df['text'].apply(clean_text)
    df['label'] = df['label'].astype(int)
    before = len(df)
    df = df[df['text'].str.len() > 0].drop_duplicates(subset=['text']).reset_index(drop=True)
    print(f'Rows: {before} to {len(df)}  (removed {before - len(df)} empty/duplicate)')
    return df[['text', 'label']]

df_clean = preprocess_dataframe(df_no_nan)
print(f'Clean dataset: {len(df_clean)} rows')
print(f'Label counts: {df_clean["label"].value_counts().rename({0: "Benign", 1: "Harmful"}).to_dict()}')

## 5. 80/20 Train-Test Split

In [ ]:
train_df, test_df = train_test_split(
    df_clean, test_size=0.20, stratify=df_clean['label'], random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

n_harmful_train = int(train_df['label'].sum())
n_harmful_test  = int(test_df['label'].sum())
print(f'Train : {len(train_df)} samples  (harmful={n_harmful_train}, benign={len(train_df)-n_harmful_train})')
print(f'Test  : {len(test_df)} samples  (harmful={n_harmful_test}, benign={len(test_df)-n_harmful_test})')

## 6. Dataset & Metrics

In [ ]:
class InjectionDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe: pd.DataFrame, tokenizer, max_length: int):
        self.texts      = dataframe['text'].tolist()
        self.labels     = dataframe['label'].tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict:
        enc = self.tokenizer(
            self.texts[idx], max_length=self.max_length,
            padding=False, truncation=True, return_tensors=None,
        )
        enc['labels'] = self.labels[idx]
        return enc


def compute_metrics(labels, preds, probs) -> Dict[str, float]:
    # Replace any NaN/Inf probs with 0.5 (neutral) before scoring
    probs = np.array(probs, dtype=np.float64)
    probs = np.where(np.isfinite(probs), probs, 0.5)
    probs = np.clip(probs, 0.0, 1.0)
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = float('nan')
    return {
        'accuracy':  accuracy_score(labels, preds),
        'precision': precision_score(labels, preds, zero_division=0),
        'recall':    recall_score(labels, preds, zero_division=0),
        'f1':        f1_score(labels, preds, zero_division=0),
        'auc':       auc,
    }


def security_score(m: Dict[str, float]) -> float:
    # If any value is NaN drop it
    # and renormalise weights so score stays in [0, 1].
    components = [
        (m['recall'],   0.40),
        (m['f1'],       0.30),
        (m['auc'],      0.20),
        (m['accuracy'], 0.10),
    ]
    valid = [(v, w) for v, w in components if v == v]
    if not valid:
        return float('nan')
    total_w = sum(w for _, w in valid)
    return sum(v * w for v, w in valid) / total_w

## 7. Training Section

In [ ]:
ID2LABEL = {0: 'Benign', 1: 'Harmful'}
LABEL2ID = {'Benign': 0, 'Harmful': 1}


def _train_epoch(model, loader, optimizer, scheduler,
                   gradient_clip: float, loss_fn=None) -> float:
    model.train()
    total = 0.0
    for batch in loader:
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}
        if loss_fn is not None:
            labels_t = batch.pop('labels')
            logits   = model(**batch).logits.float()
            loss     = loss_fn(logits, labels_t)
        else:
            loss = model(**batch).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
        total += loss.item()
    return total / len(loader)

@torch.no_grad()

def _eval(model, loader) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    for batch in loader:
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}
        # We cast to fp32 in order to avoid NaN from bf16 overflow
        logits = model(**batch).logits.float()
        # We convert data to CPU in order to calculate NumPy
        all_probs.extend(torch.softmax(logits, -1)[:, 1].cpu().numpy())
        all_preds.extend(logits.argmax(-1).cpu().numpy())
        all_labels.extend(batch['labels'].cpu().numpy())
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)


def run_training(
    model_name:    str,

    # 7 hyperparameters
    max_length:    int,
    batch_size:    int,
    num_epochs:    int,
    learning_rate: float,
    weight_decay:  float,
    warmup_ratio:  float,
    gradient_clip: float,

    train_df: pd.DataFrame,
    test_df:  pd.DataFrame,
    verbose:  bool = False,
) -> Dict:
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    collator  = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors='pt')

    train_loader = DataLoader(
        InjectionDataset(train_df, tokenizer, max_length),
        batch_size=batch_size, shuffle=True, collate_fn=collator, num_workers=0)
    test_loader = DataLoader(
        InjectionDataset(test_df, tokenizer, max_length),
        batch_size=batch_size * 2, shuffle=False, collate_fn=collator, num_workers=0)

    for _noisy in ('transformers.safetensors_conversion',
                   'huggingface_hub.utils._http',
                   'httpcore', 'httpx'):
        _logging.getLogger(_noisy).setLevel(_logging.CRITICAL)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2,
        id2label=ID2LABEL, label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    ).to(DEVICE)

    optimizer    = AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    total_steps  = len(train_loader) * num_epochs
    scheduler    = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * warmup_ratio),
        num_training_steps=total_steps)

    # Weighted cross-entropy to handle class imbalance
    # pos_weight = n_negative / n_positive
    labels_arr   = np.array(train_df['label'].tolist())
    n_pos        = max(labels_arr.sum(), 1)
    n_neg        = max(len(labels_arr) - n_pos, 1)
    pos_weight   = min(n_neg / n_pos, 10.0)
    pw_tensor    = torch.tensor([pos_weight], dtype=torch.float32, device=DEVICE)
    loss_fn      = torch.nn.BCEWithLogitsLoss(pos_weight=pw_tensor)

    def weighted_loss(logits, labels):
        return loss_fn(logits[:, 1] - logits[:, 0], labels.float())

    history = []
    best_f1, best_state = 0.0, None

    for epoch in range(1, num_epochs + 1):
        avg_loss = _train_epoch(model, train_loader, optimizer, scheduler,
                                gradient_clip, loss_fn=weighted_loss)
        labels, preds, probs = _eval(model, test_loader)
        m = compute_metrics(labels, preds, probs)
        history.append({'epoch': epoch, 'loss': avg_loss, **m})
        if verbose:
            print(f'    ep{epoch}  loss={avg_loss:.4f}  rec={m["recall"]:.4f}  '
                  f'f1={m["f1"]:.4f}  auc={m["auc"]:.4f}')
        if m['f1'] > best_f1:
            best_f1    = m['f1']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    final_labels, final_preds, final_probs = _eval(model, test_loader)
    final_metrics = compute_metrics(final_labels, final_preds, final_probs)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    hparams = dict(
        max_length=max_length, batch_size=batch_size, num_epochs=num_epochs,
        learning_rate=learning_rate, weight_decay=weight_decay,
        warmup_ratio=warmup_ratio, gradient_clip=gradient_clip,
    )
    return {
        'metrics':    final_metrics,
        'history':    history,
        'labels':     final_labels,
        'preds':      final_preds,
        'probs':      final_probs,
        'hparams':    hparams,
        'sec_score':  security_score(final_metrics),
        'cls_report': classification_report(
            final_labels, final_preds,
            target_names=['Benign', 'Harmful'], digits=4),
    }

## 8. DistilBERT Model Training

In [ ]:

# We train DistilBERT with optimized hyperparameters.

HP_SEARCH_RESULTS: Dict[str, List[Dict]] = {}   # results per model

for model_label, grid in MODEL_HP_GRIDS.items():
    model_name = MODEL_HF_IDS[model_label]
    keys   = list(grid.keys())
    combos = list(itertools.product(*grid.values()))

    print(f'\n{"="*72}')
    print(f'  TRAINING: {model_label}  ({model_name})')
    print(f'{"="*72}')

    runs: List[Dict] = []

    for i, combo in enumerate(combos, 1):
        hp = dict(zip(keys, combo))
        print(
            f'  [{i:3d}/{len(combos)}]  '
            f'lr={hp["learning_rate"]:.0e}  '
            f'wd={hp["weight_decay"]}  '
            f'wu={hp["warmup_ratio"]}  '
            f'ep={hp["num_epochs"]}  '
            f'bs={hp["batch_size"]}  '
            f'ml={hp["max_length"]}  '
            f'gc={hp["gradient_clip"]}',
            end='  … ',
        )
        try:
            result = run_training(
                model_name    = model_name,
                max_length    = hp['max_length'],
                batch_size    = hp['batch_size'],
                num_epochs    = hp['num_epochs'],
                learning_rate = hp['learning_rate'],
                weight_decay  = hp['weight_decay'],
                warmup_ratio  = hp['warmup_ratio'],
                gradient_clip = hp['gradient_clip'],
                train_df      = train_df,
                test_df       = test_df,
                verbose       = True,
            )
            runs.append(result)
            m = result['metrics']
            print(f'sec={result["sec_score"]:.4f}  '
                  f'rec={m["recall"]:.4f}  f1={m["f1"]:.4f}')
        except Exception as e:
            print(f'FAILED: {e}')

    HP_SEARCH_RESULTS[model_label] = runs
    if runs:
        best = max(runs, key=lambda r: r['sec_score'])
        print(f'\nDistilBERT training complete: sec={best["sec_score"]:.4f}  '
              f'HP={best["hparams"]}')

print('\nTraining complete.\n')


## 9. DistilBERT Evaluation Results

In [ ]:
BEST_RESULTS: Dict[str, Dict] = {}

for model_label, runs in HP_SEARCH_RESULTS.items():
    if not runs:
        print(f'{model_label}: no successful runs')
        continue

    rows = []
    for r in runs:
        row = {**r['hparams']}
        row.update({k.capitalize(): round(v, 4) for k, v in r['metrics'].items()})
        row['Security Score'] = round(r['sec_score'], 4)
        rows.append(row)

    df_hp = pd.DataFrame(rows).sort_values('Security Score', ascending=False)

    print(f'\n{"="*100}')
    print(f'  {model_label} — Evaluation Results')
    print(f'{"="*100}')
    print(df_hp.to_string(index=False))
    print(f'{"="*100}')

    fname = f'Results_{model_label.lower().replace("-", "_")}.csv'
    BEST_RESULTS[model_label] = max(runs, key=lambda r: r['sec_score'])
